### `privileged_command_events`
Admin/ops system.

| Field | Type | Notes |
|---|---|---|
| event_id | UUID (PK) | |
| system_identifier | string | service_account_name / adm-name |
| command_timestamp | timestamp | |
| command | string | Exact command text |
| command_category | string | config_change / user_management / data_deletion / service_control / log_management / access_control |
| resource_id | string (FK → resources), nullable | Populated only when target is a catalog resource |
| target_description | string, nullable | Free text, used when target isn't a catalog resource (e.g. "entity:E047 permissions", "deployment_server") |

Frequency:

Human: 0/1/2 commands with weights 0.75/0.20/0.05
ci-cd: 1-4, security-scanning: 1-3, backup-automation: 1-2

Category weighting:

Human: uniform across all 6
ci-cd: service_control 50% / config_change 40% / data_deletion 10%
security-scanning: log_management 40% / access_control 40% / user_management 20%
backup-automation: data_deletion 40% / config_change 30% / log_management 30%

Command pool: the 18-command table above, each tied to either resource_id (a specific catalog resource) or target_description (free text, with the three permission-related commands referencing a random other entity_id)

Timing: human — working-hours window (home timezone); non-human — random throughout the day, same pattern as file_access

In [0]:
pip install pycountry

In [0]:
import uuid
import datetime
import random
import pycountry
import pytz
import zoneinfo
from pyspark.sql.functions import col, collect_set

In [0]:
# List of 44 european countries
europe_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", 
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", 
    "Denmark", "Estonia", "Finland", "France", "Germany", 
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", 
    "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", 
    "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia", 
    "Norway", "Poland", "Portugal", "Romania", "Russia", 
    "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", 
    "Sweden", "Switzerland", "Ukraine", "United Kingdom"
]

# Human frequecy seps
human_frequency_list = [0, 1, 2]
human_frequency_weights = [0.75, 0.20, 0.05]

# Command category specs
cmd_category_list = {
    "config_change": [
        ("update_firewall_rules", "target_description", "firewall_config"),
        ("modify_service_config", "target_description", "deployment_server"),
        ("update_environment_variables", "resource_id", "deployment_pipeline")
    ],
    "user_management": [
        ("grant_user_permissions", "target_description", "entity:{other_entity_id} permissions"),
        ("revoke_user_access", "target_description", "entity:{other_entity_id} permissions"),
        ("create_service_account", "target_description", "new service account provisioning")
    ],
    "data_deletion": [
        ("purge_old_backups", "target_description", "backup_archive"),
        ("delete_expired_logs", "resource_id", "security_audit_logs"),
        ("remove_stale_records", "resource_id", "customer_pii_store")
    ],
    "service_control": [
        ("restart_service", "target_description", "deployment_server"),
        ("deploy_release", "resource_id", "deployment_pipeline"),
        ("scale_instance_count", "target_description", "compute_cluster")
    ],
    "log_management": [
        ("archive_logs", "resource_id", "security_audit_logs"),
        ("export_audit_trail", "resource_id", "security_audit_logs"),
        ("rotate_log_files", "target_description", "logging_system")
    ],
    "access_control": [
        ("modify_access_policy", "resource_id", "admin_credentials_vault"),
        ("update_role_permissions", "target_description", "entity:{other_entity_id} permissions"),
        ("disable_mfa_requirement", "target_description", "auth_policy")
    ]
}

ci_cd_list = ["service_control", "config_change", "data_deletion"]
ci_cd_weights = [0.5, 0.4, 0.1]
security_scanning_list = ["log_management", "access_control", "user_management"]
security_scanning_weights = [0.4, 0.4, 0.2]
backup_automation_list = ["data_deletion", "config_change", "log_management"]
backup_automation_weights = [0.4, 0.3, 0.3]

In [0]:
entity_df = spark.read.table("entity_risk_platform.seed_data.entities").select("entity_id", "entity_type", "role", "home_country")

command_identifier_df = spark.read.table("entity_risk_platform.seed_data.entity_system_identifiers").filter(col("system_name") == "privileged_command")

resouce_df = spark.read.table("entity_risk_platform.seed_data.resources")

command_entity_df = command_identifier_df.join(entity_df, "entity_id")

# command_entity_df.show()

command_entities = [row.asDict() for row in command_entity_df.collect()]

resources = {row["resource_name"]: row["resource_id"] for row in resouce_df.collect()}

In [0]:
europe_timzones = {}

for ec in europe_countries:
    country = pycountry.countries.search_fuzzy(ec)[0]
    country_code = country.alpha_2
    country_timezone = pytz.country_timezones.get(country_code)[0]
    europe_timzones[ec] = country_timezone

# print(europe_timzones)

In [0]:
# chosen_entity = random.choice(command_entities)
# print(chosen_entity)

In [0]:
def generate_command_event_count(entity):
    event_count = 0
    if(entity["entity_type"] == "human"):
        event_count = random.choices(human_frequency_list, weights=human_frequency_weights)[0]
    elif(entity["entity_type"] in ["service_account", "agent"]):
        if(entity["role"] == "ci-cd"):
            event_count = random.randrange(1, 5)
        elif(entity["role"] == "security-scanning"):
            event_count = random.randrange(1, 4)
        elif(entity["role"] == "backup-automation"):
            event_count = random.randrange(1, 3)

    return event_count

In [0]:
def generate_timestamp(date, timezone, min_sec, max_sec):
    random_date = datetime.datetime.combine(date.date(), datetime.time.min) + datetime.timedelta(seconds=random.randrange(min_sec, max_sec))
    local_date = random_date.replace(tzinfo=zoneinfo.ZoneInfo(timezone))
    utc_date = local_date.astimezone(zoneinfo.ZoneInfo("UTC"))
    return utc_date

# print(generate_timestamp(datetime.datetime.now(), chosen_entity["home_country"], 0, 3600))

In [0]:
def generate_command_timestamp(date, entity):
    if(entity["entity_type"] == "human"):
        # 8 AM
        min_seconds = 8 * 3600 
        # 6 PM
        max_seconds = 18 * 3600
        return generate_timestamp(date, europe_timzones[entity["home_country"]], min_seconds, max_seconds)
    else:
        # 12 AM
        min_seconds = 0
        # 12 PM
        max_seconds = 24 * 3600
        return generate_timestamp(date, "UTC", min_seconds, max_seconds)

In [0]:
def generate_cmd(entity):
    if(entity["entity_type"] == "human"):
        category = random.choice(list(cmd_category_list.keys()))
    elif(entity["entity_type"] in ["service_account", "agent"]):
        if(entity["role"] == "ci-cd"):
            category = random.choices(ci_cd_list, weights=ci_cd_weights)[0]
        elif(entity["role"] == "security-scanning"):
            category = random.choices(security_scanning_list, weights=security_scanning_weights)[0]
        elif(entity["role"] == "backup-automation"):
            category = random.choices(backup_automation_list, weights=backup_automation_weights)[0]
    
    cmd_exc = random.choice(cmd_category_list[category])
    command = cmd_exc[0]
    if(cmd_exc[1] == "resource_id"):
        resource = resources[cmd_exc[2]]
        description = None
    elif(cmd_exc[1] == "target_description"):
        if("{other_entity_id}" in cmd_exc[2]):
            random_entity_id = random.choice([item["entity_id"] for item in command_entities if item["entity_id"] != entity["entity_id"]])
            description = cmd_exc[2].format(other_entity_id=random_entity_id)
        else:
            description = cmd_exc[2]
        resource = None

    return command, category, resource, description

In [0]:
target_date = datetime.datetime.now()

In [0]:
command_event_list = []

In [0]:
def generate_command_events(date, entities):
    events = []
    for chosen_entity in entities:
        command_events = generate_command_event_count(chosen_entity)
        for _ in range(command_events):
            command, category, resource, description = generate_cmd(chosen_entity)
            events.append({
                "event_id": str(uuid.uuid4()),
                "system_identifier": chosen_entity["system_identifier"],
                "command_timestamp": generate_command_timestamp(date, chosen_entity),
                "command": command,
                "command_category": category,
                "resource_id": resource,
                "target_description": description
            })
    return events

command_event_list = generate_command_events(target_date, command_entities)
print(command_event_list)